# PHASE 1 — MACHINE LEARNING FOUNDATIONS


# Day 05 — Pipelines


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Construct a `Pipeline` to chain multiple Scikit-learn steps together.
- Explain how Pipelines automatically prevent data leakage.
- Explain how Pipelines prevent training/inference mismatch.
- Simplify messy ML code into a clean, elegant workflow.


## 2. Prerequisites
- Day 2 (Scikit-learn API).
- Day 4 (Preprocessing: Imputation and Scaling).


## 3. Concept
A **Pipeline** sequentially applies a list of transformers and a final estimator. 
Instead of manually saving an Imputer, manually calling `imputer.transform(X)`, then manually saving a Scaler, and manually calling `scaler.transform(X)`, a Pipeline bundles all these steps into a single object.

You treat the entire Pipeline as if it were a single model. You call `pipeline.fit(X, y)` and it automatically executes every step in the correct order.


## 4. Why Does This Matter?
Pipelines are arguably the most important feature in Scikit-learn for production ML because they prevent three massive problems:
1. **Data Leakage**: During cross-validation, pipelines ensure that scaling/imputing happens *after* the split on every single fold.
2. **Inconsistent Preprocessing**: Doing step A then step B in training, but accidentally doing step B then step A in testing.
3. **Training/Inference Mismatch**: Deploying the model but forgetting to deploy the exact scaler object alongside it. A Pipeline bundles everything together into one deployable artifact.


## 5. Intuition
Think of an assembly line in a car factory.
- Step 1 (Imputer): Put on the wheels.
- Step 2 (Scaler): Paint the car.
- Step 3 (Model): Drive the car.

Without an assembly line (Pipeline), you have to manually carry the car from station to station. If you forget to paint the test car before trying to drive it, the car fails inspection. The Pipeline automates the conveyor belt.


## 6. Mathematical Foundation
Mathematically, a Pipeline represents function composition. If we have an Imputer $I$, a Scaler $S$, and a Model $M$, predicting $\hat{y}$ on raw data $X$ requires:

$$ \hat{y} = M(S(I(X))) $$

The Pipeline abstracts this so you only ever have to compute:

$$ \hat{y} = Pipeline(X) $$


## 7. Scikit-learn API
Using `Pipeline` from `sklearn.pipeline` requires passing a list of tuples. Each tuple contains a string name (that you make up) and the estimator object:

```python
Pipeline([
    ('step1_name', Transformer1()),
    ('step2_name', Transformer2()),
    ('model_name', Estimator())
])
```


## 8. Simple Example
Let's recreate yesterday's numeric processing (Imputation + Scaling) combined with a Logistic Regression model, but this time using a Pipeline.


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Create messy data
X = np.array([[25.0], [np.nan], [30.0], [45.0], [50.0], [np.nan], [22.0], [60.0]])
y = np.array([0, 1, 0, 1, 1, 0, 0, 1])

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 3. Build the Pipeline
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')), # Step 1
    ('scaler', StandardScaler()),              # Step 2
    ('classifier', LogisticRegression())       # Step 3
])

# 4. Fit the ENTIRE pipeline at once
pipe.fit(X_train, y_train)

# 5. Predict using the ENTIRE pipeline
predictions = pipe.predict(X_test)
print('Predictions:', predictions)


## 9. Code Walkthrough
- `pipe = Pipeline(...)`: We define the chronological steps of our workflow.
- `pipe.fit(X_train, y_train)`: Under the hood, this does:
  1. `X_imp = imputer.fit_transform(X_train)`
  2. `X_scaled = scaler.fit_transform(X_imp)`
  3. `classifier.fit(X_scaled, y_train)`
- `pipe.predict(X_test)`: Under the hood, this does:
  1. `X_imp = imputer.transform(X_test)` (Note: ONLY `.transform()`!)
  2. `X_scaled = scaler.transform(X_imp)`
  3. `return classifier.predict(X_scaled)`
  
Notice how the Pipeline safely protects the test set from `.fit_transform()` leakage automatically!


## 10. Experiment
We can access individual steps inside the pipeline using the `.named_steps` dictionary. Let's look at what the imputer learned.


In [ ]:
learned_mean = pipe.named_steps['imputer'].statistics_
print('The imputer learned this mean from the training data:', learned_mean)

learned_coef = pipe.named_steps['classifier'].coef_
print('The logistic regression learned this coefficient:', learned_coef)


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
from sklearn.pipeline import make_pipeline
pipe2 = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression()
)


> **Question:** What is the difference between `Pipeline` and `make_pipeline`? If we didn't provide string names in `make_pipeline`, what name will it assign to `StandardScaler`? 

**Think before running the next cell!**


In [ ]:
print(list(pipe2.named_steps.keys()))
print('\nWhy? `make_pipeline` is a shortcut that automatically generates names for the steps by lowercasing the class name (e.g., StandardScaler -> standardscaler).')


## 12. Coding Exercise
Build a pipeline using `make_pipeline` that:
1. Fills missing values with the constant value `-1`.
2. Uses `MinMaxScaler`.
3. Trains a `DecisionTreeClassifier`.
Fit it on `X_train` and score it on `X_test`.


In [ ]:
# YOUR CODE HERE
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler

my_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=-1),
    MinMaxScaler(),
    DecisionTreeClassifier(random_state=42)
)
my_pipe.fit(X_train, y_train)
score = my_pipe.score(X_test, y_test)
print('Pipeline Score:', score)


## 13. Debugging Challenge
The pipeline below crashes with a `TypeError`. Why?


In [ ]:
# Buggy code
try:
    bad_pipe = Pipeline([
        ('model', LogisticRegression()),
        ('scaler', StandardScaler())
    ])
    # bad_pipe.fit(X_train, y_train) # This would crash
except Exception as e:
    print('Error:', e)


> **Hint:** Look at the order of the steps. What does `LogisticRegression` output? Can a Scaler scale the output of a predictor? (No, the predictor must always be the **last** step in the Pipeline).


## 14. Model Evaluation
When we score a pipeline (`pipe.score(X, y)`), the pipeline pushes `X` through all the transformers, and then calls the `.score()` method of the final estimator. It is perfectly identical to evaluating the model on manually preprocessed data, just much safer.


## 15. Real-World Example
In production, you serialize (save) the entire `Pipeline` object using a library like `joblib`. 
When the Web API receives new user data, it simply calls `loaded_pipeline.predict(user_data)`. 
If you didn't use a pipeline, your backend software engineer would have to perfectly rewrite your Imputation and Scaling logic in the web server code, which almost always introduces bugs!


## 16. Mini Project
Can we pass data directly to a pipeline without train_test_split? Yes, but you shouldn't if you want to evaluate it. Let's demonstrate that `pipe.predict()` works on a single raw data point.


In [ ]:
# Raw data point (a single user input)
new_user_data = np.array([[np.nan]])

# Pipeline handles the imputation, scaling, and prediction automatically!
prediction = pipe.predict(new_user_data)
print('Prediction for new user with missing data:', prediction)


## 17. Common Mistakes
- **Putting the model first**: The estimator/model must ALWAYS be the final step in a Pipeline.
- **Forgetting parentheses**: Writing `Pipeline([('scaler', StandardScaler)])` instead of `StandardScaler()`. You must instantiate the objects.
- **Calling fit_transform on a Pipeline**: `pipe.fit_transform(X)` is usually an error because predictors (the final step) don't have a `.transform()` method (they have `.predict()`). Just call `pipe.fit(X, y)`.


## 18. Interview Questions
- **Beginner**: What is the purpose of a Scikit-learn Pipeline?
- **Intermediate**: Explain how a Pipeline prevents data leakage during cross-validation.
- **Advanced**: How does a Pipeline's `.fit()` method differ mechanically from its `.predict()` method regarding the intermediate steps?


## 19. Knowledge Check
- What is the shortcut function to build a Pipeline without naming the steps? (`make_pipeline`)
- Which step in the pipeline is allowed to not have a `.transform()` method? (The final estimator step)


## 20. Summary
- **Pipelines** bundle transformers and a final predictor into one object.
- They automate calling `.fit_transform()` on training data and `.transform()` on testing data.
- They are the ultimate defense against data leakage and preprocessing mismatch.
- Use `make_pipeline` for rapid prototyping.


## 21. Homework
Load a dataset of your choice. Build a pipeline with `SimpleImputer`, `StandardScaler`, and `KNeighborsClassifier`. Train it and score it. Then inspect `.named_steps` to view the fitted K-Neighbors model.
